# IMDb: 100 mejores pel?culas

Este cuaderno implementa un flujo reproducible: **leer ? inspeccionar y validar ? limpiar y transformar ? analizar ? comunicar resultados**. El CSV y el JSON representan las mismas 100 pel?culas; el JSON se gener? a partir del CSV para demostrar la lectura de ambos formatos.

## C?mo ejecutarlo

1. Abr? esta carpeta como proyecto en VS Code y asegurate de tener `pandas` instalado (`%pip install pandas` si hace falta).
2. Ejecut? **Run All / Ejecutar todo**, de arriba hacia abajo. No ejecutes celdas fuera de orden.
3. Para cambiar el archivo de entrada, modific? `FORMATO_ENTRADA` en la siguiente celda (`"csv"` o `"json"`). En cada ejecuci?n se leen, validan y comparan ambos archivos.
4. Para cambiar la b?squeda, edit? `CRITERIO_BUSQUEDA` y `VALOR_BUSQUEDA` en la ?ltima celda. El resultado se muestra y tambi?n se guarda en `resultado_busqueda.csv`.

## Decisiones de limpieza

- Se descarta `Titulo_EN` porque el alcance del proyecto es buscar por t?tulo en espa?ol; los archivos originales no se modifican.
- Se quitan espacios sobrantes de los textos, se normalizan los espacios de los g?neros y se convierten ranking, a?o y nota a tipos num?ricos.
- Los encabezados finales se renombran como `Ranking`, `Titulo`, `A?o` y `Nota` para facilitar su lectura.
- No se descartan duplicados ni se inventan o reemplazan datos. Si faltan columnas, hay valores vac?os o inv?lidos, rankings repetidos o valores fuera de rango, el flujo se detiene indicando el problema.
- `resultado_busqueda.csv` es una salida generada por el an?lisis, no una fuente de entrada.

In [ ]:
from pathlib import Path
import pandas as pd

RUTA_CSV = Path("IMDb 100 Mejores Peliculas.csv")
RUTA_JSON = Path("IMDb 100 Mejores Peliculas.json")
FORMATO_ENTRADA = "csv"  # Cambiar a "json" para seleccionar la otra fuente.


def cargar_peliculas(ruta):
    """Carga pel?culas desde un archivo CSV o JSON seg?n su extensi?n."""
    if not ruta.is_file():
        raise FileNotFoundError(f"No se encontr? el archivo de datos: {ruta}")
    extension = ruta.suffix.lower()
    if extension == ".csv":
        return pd.read_csv(ruta)
    if extension == ".json":
        return pd.read_json(ruta)
    raise ValueError(f"Formato no admitido para {ruta.name}: {extension}")


if FORMATO_ENTRADA not in {"csv", "json"}:
    raise ValueError('FORMATO_ENTRADA debe ser "csv" o "json".')

peliculas_csv_originales = cargar_peliculas(RUTA_CSV)
peliculas_json_originales = cargar_peliculas(RUTA_JSON)
peliculas_originales = (
    peliculas_csv_originales if FORMATO_ENTRADA == "csv" else peliculas_json_originales
)

print(f"Formato elegido para el an?lisis: {FORMATO_ENTRADA.upper()}")
print(f"CSV: {peliculas_csv_originales.shape[0]} filas, {peliculas_csv_originales.shape[1]} columnas")
print(f"JSON: {peliculas_json_originales.shape[0]} filas, {peliculas_json_originales.shape[1]} columnas")

In [ ]:
COLUMNAS_ESPERADAS = [
    "Rank", "Titulo_ES", "Titulo_EN", "Anio", "Nota_IMDb", "Director", "Genero"
]
COLUMNAS_NUMERICAS = ["Rank", "Anio", "Nota_IMDb"]
COLUMNAS_TEXTO = ["Titulo_ES", "Titulo_EN", "Director", "Genero"]


def inspeccionar_y_validar(datos, nombre):
    """Comprueba esquema, valores, duplicados y rangos antes del an?lisis."""
    faltantes = [columna for columna in COLUMNAS_ESPERADAS if columna not in datos.columns]
    if faltantes:
        raise ValueError(f"{nombre}: faltan columnas requeridas: {faltantes}")
    datos = datos[COLUMNAS_ESPERADAS].copy()
    problemas = []

    vacios = datos[COLUMNAS_ESPERADAS].isna().sum()
    vacios_texto = datos[COLUMNAS_TEXTO].apply(
        lambda columna: columna.astype("string").str.strip().eq("").sum()
    )
    if vacios.sum() or vacios_texto.sum():
        problemas.append(
            f"hay valores nulos ({vacios.to_dict()}) o textos vac?os ({vacios_texto.to_dict()})"
        )

    for columna in COLUMNAS_NUMERICAS:
        convertidos = pd.to_numeric(datos[columna], errors="coerce")
        invalidos = convertidos.isna()
        if invalidos.any():
            problemas.append(f"{columna} contiene {int(invalidos.sum())} valores no num?ricos")
        if columna in {"Rank", "Anio"} and not convertidos.dropna().mod(1).eq(0).all():
            problemas.append(f"{columna} debe contener n?meros enteros")
        datos[columna] = convertidos

    if datos["Rank"].duplicated().any():
        problemas.append("hay valores de Rank repetidos")
    if len(datos) != 100:
        problemas.append(f"se esperaban 100 filas y hay {len(datos)}")
    if not datos["Rank"].between(1, 100).all():
        problemas.append("hay rankings fuera del rango 1?100")
    if not datos["Anio"].between(1888, 2100).all():
        problemas.append("hay a?os fuera del rango razonable 1888?2100")
    if not datos["Nota_IMDb"].between(0, 10).all():
        problemas.append("hay notas fuera del rango 0?10")
    if datos.duplicated().any():
        problemas.append("hay filas completamente duplicadas")
    if problemas:
        raise ValueError(f"{nombre}: validaci?n fallida: " + "; ".join(problemas))

    print(f"{nombre}: esquema correcto; 100 filas; sin vac?os ni duplicados; rangos v?lidos.")
    return datos


def limpiar_y_transformar(datos):
    """Normaliza texto y tipos, y conserva las columnas definidas para el proyecto."""
    resultado = datos.copy()
    for columna in COLUMNAS_TEXTO:
        resultado[columna] = resultado[columna].str.strip()
    resultado["Genero"] = resultado["Genero"].apply(
        lambda valor: ", ".join(genero.strip() for genero in valor.split(","))
    )
    resultado["Rank"] = resultado["Rank"].astype("int64")
    resultado["Anio"] = resultado["Anio"].astype("int64")
    resultado["Nota_IMDb"] = resultado["Nota_IMDb"].astype("float64")
    resultado = resultado.drop(columns=["Titulo_EN"]).rename(
        columns={"Rank": "Ranking", "Titulo_ES": "Titulo", "Anio": "A?o", "Nota_IMDb": "Nota"}
    )
    return resultado.sort_values("Ranking").reset_index(drop=True)


print("Columnas y tipos del archivo seleccionado:")
print(peliculas_originales.dtypes)
csv_validado = inspeccionar_y_validar(peliculas_csv_originales, "CSV")
json_validado = inspeccionar_y_validar(peliculas_json_originales, "JSON")
peliculas_csv = limpiar_y_transformar(csv_validado)
peliculas_json = limpiar_y_transformar(json_validado)
pd.testing.assert_frame_equal(peliculas_csv, peliculas_json, check_dtype=False)
print("CSV y JSON equivalentes despu?s de la limpieza.")
peliculas = peliculas_csv if FORMATO_ENTRADA == "csv" else peliculas_json
print("Columnas finales para el an?lisis:", peliculas.columns.tolist())
display(peliculas.head())

In [ ]:
def buscar_peliculas(datos, criterio, valor):
    """Filtra por ranking, a?o, t?tulo en espa?ol, director o g?nero."""
    columnas_texto = {"titulo": "Titulo", "director": "Director", "genero": "Genero"}
    if criterio == "ranking":
        columna = "Ranking"
    elif criterio == "a?o":
        columna = "A?o"
    elif criterio in columnas_texto:
        texto = str(valor).strip()
        if not texto:
            raise ValueError("El texto de b?squeda no puede estar vac?o.")
        columna = columnas_texto[criterio]
        coincidencias = datos[columna].str.contains(texto, case=False, regex=False, na=False)
        return datos.loc[coincidencias].sort_values("Ranking").reset_index(drop=True)
    else:
        opciones = ["ranking", "a?o", "titulo", "director", "genero"]
        raise ValueError(f"Criterio no v?lido: {criterio}. Eleg? uno de {opciones}.")
    try:
        valor_numerico = int(valor)
    except (TypeError, ValueError) as error:
        raise ValueError(f"Para buscar por {criterio}, ingres? un n?mero entero.") from error
    return datos.loc[datos[columna] == valor_numerico].sort_values("Ranking").reset_index(drop=True)


CRITERIO_BUSQUEDA = "genero"  # ranking, a?o, titulo, director o genero
VALOR_BUSQUEDA = "Drama"
resultado_csv = buscar_peliculas(peliculas_csv, CRITERIO_BUSQUEDA, VALOR_BUSQUEDA)
resultado_json = buscar_peliculas(peliculas_json, CRITERIO_BUSQUEDA, VALOR_BUSQUEDA)
pd.testing.assert_frame_equal(resultado_csv, resultado_json, check_dtype=False)
resultado = resultado_csv if FORMATO_ENTRADA == "csv" else resultado_json
print(f"B?squeda: {CRITERIO_BUSQUEDA} = {VALOR_BUSQUEDA!r}")
print(f"Pel?culas encontradas: {len(resultado)}")
if resultado.empty:
    print("No se encontraron pel?culas con ese criterio.")
else:
    display(resultado)
RUTA_RESULTADO = Path("resultado_busqueda.csv")
resultado.to_csv(RUTA_RESULTADO, index=False, encoding="utf-8-sig")
print(f"Resultado guardado en: {RUTA_RESULTADO.resolve()}")
print("La misma b?squeda produjo resultados equivalentes desde CSV y JSON.")